#Take-home exercise 02
Chunlin An | ca2965

In [0]:
from pyspark.sql.functions import col, year, current_date, datediff, round, avg, min, max, rank, upper, substring, when, isnull, concat, udf, create_map, lit, floor, sum, countDistinct, count
from itertools import chain
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
import re


In [0]:
# read in data
df_laptimes = spark.read.csv('/Volumes/gr5069/raw/f1_data/lap_times.csv', header = True)

df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header = True)

df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv', header = True)

df_races = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv', header = True)

##1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.


I first joined the drivers dataframe and pitstops dataframe on the driverID, then grouped by driver and race ID to obtain race-specific data for each driver. I converted milliseconds to seconds for readability.
I then calculated the mean stopping time using the `milliseconds` column (cast as double), stored in a separate dataframe with driverRef, race ID, and average duration.
For fastest and slowest pit stops, I grouped by race ID only so they represent the fastest/slowest pit stops across all drivers in each race. 
Finally, I joined the two dataframes on race ID, then sorted by race ID and average pit stop time, so that the final dataframe displays data in order of race ID, with avg_pitstop_time ranking from fastest to slowest.


In [0]:
# join dataframes

df_pit_drivers = df_drivers.select('driverId', 'driverRef').join(df_pitstops, on = ['driverId'])

df_pit_drivers.show(5)

+--------+------------------+------+----+---+--------+--------+------------+
|driverId|         driverRef|raceId|stop|lap|    time|duration|milliseconds|
+--------+------------------+------+----+---+--------+--------+------------+
|     153|       alguersuari|   841|   1|  1|17:05:23|  26.898|       26898|
|      30|michael_schumacher|   841|   1|  1|17:05:52|  25.021|       25021|
|      17|            webber|   841|   1| 11|17:20:48|  23.426|       23426|
|       4|            alonso|   841|   1| 12|17:22:34|  23.251|       23251|
|      13|             massa|   841|   1| 13|17:24:10|  23.842|       23842|
+--------+------------------+------+----+---+--------+--------+------------+
only showing top 5 rows


In the code above I joined the drivers dataframe with the pit stop dataframe on driver ID (as it is one of the columns shared by the two dataframes).

In [0]:
# cast duration time (milliseconds) as double, then convert to seconds for clarity
df_clean = df_pit_drivers.withColumn("duration_seconds", round(col("milliseconds").cast("double")/1000, 2))

# group by driver id, race id and calculate average pit stop time
df_avg_pit = df_clean.groupBy("driverRef", "raceId").agg(
    avg("duration_seconds").alias("avg_pitstop_time"))

df_avg_pit.show(5)

+---------+------+----------------+
|driverRef|raceId|avg_pitstop_time|
+---------+------+----------------+
|    resta|   841|          24.595|
|   trulli|   842|           25.52|
|    glock|   843|           26.14|
|   webber|   846|          32.005|
|   petrov|   846|           28.86|
+---------+------+----------------+
only showing top 5 rows


Here I made a copy of the original data and named it 'df_cleaned'. I seleced the 'milliseconds' column representing the duration in the pit, casted it as a double with `cast("double")`, divided by 1000 so that the data is more interpretable in seconds. 
I grouped the cleaned dataframe by driverRef (more interpretable than driver ID) and race ID, then added a column representing average pit stop time, calculated from taking the average of the duration column (in seconds) with `.agg(avg())`.

In [0]:
# calculate fastest vs. slowest pit stops each race
df_race_stats = df_clean.groupBy("raceId").agg(
    min("duration_seconds").alias("fastest_pitstop"),
    max("duration_seconds").alias("slowest_pitstop")
)

# join with df_avg_pit 
df_final = df_avg_pit.join(df_race_stats, on="raceId")

# sort by race ID and average pitstop time
# final dataframe displays data in order of race ID, with avg_pitstop_time ranking from fastest to slowest
df_final = df_final.orderBy("raceId", "avg_pitstop_time")

df_final.show(5)

+------+---------+----------------+---------------+---------------+
|raceId|driverRef|avg_pitstop_time|fastest_pitstop|slowest_pitstop|
+------+---------+----------------+---------------+---------------+
|  1000|   stroll|           21.29|          21.29|          25.13|
|  1000|   bottas|           21.34|          21.29|          25.13|
|  1000|ricciardo|           21.36|          21.29|          25.13|
|  1000| hamilton|           21.48|          21.29|          25.13|
|  1000| sirotkin|           21.51|          21.29|          25.13|
+------+---------+----------------+---------------+---------------+
only showing top 5 rows


Here I created a separate copy of the cleaned dataframe, grouping by only the race ID, ignoring the driver, then created two more columns representing the fastest and slowest pit stops in each race with `.min()` and `.max()` functions.
Technically I can keep the two dataframes separate, but to make the information more compact, I joined the two datasets again on `raceId`, then sorted by the race id first, then average pitstop time, so that the final dataframe displays the rows grouped by race ID and the average pitstop time column is sorted from the fasted to the slowest, by driver. The fastest and slowest pitstop columns correspond to the race ID value hence are redundant for each race.

##2.Rank order by finishing position the average time spent at the pit stop in each race.



The goal is to find each driver's finishing position in each race, then link it to their average pit stop time, and finally rank drivers within each race by their finishing position. The finishing position comes from df_laptimes, specifically the position on the last lap of each race (since that reflects the final result). I then joined this with the average pit stop times calculated in Q1, and ranked by finishing position within each race.


In [0]:
# find the last lap for each driver from laptimes
w_last_lap = Window.partitionBy("raceId", "driverId")

# add max lap column representing the last lap for each driver, then filter to only include the last lap
# add finishing position column renamed from 'position' to 'finishing_position'
df_last_lap = df_laptimes.withColumn("max_lap", max("lap").over(w_last_lap)).filter(
    col("lap") == col("max_lap")).select(
        "raceId", "driverId", "position").withColumnRenamed(
            "position", "finishing_position")

df_last_lap.show(5)

+------+--------+------------------+
|raceId|driverId|finishing_position|
+------+--------+------------------+
|     1|       1|                 9|
|     1|      13|                 3|
|     1|      18|                 1|
|     1|       7|                16|
|    10|       6|                 9|
+------+--------+------------------+
only showing top 5 rows


df_laptimes tracks position lap-by-lap, so the finishing position is the position value on the last lap. I used a `Window` partitioned by race ID and driver ID. Spark groups rows by unique race-driver combinations, so all laps for a given driver in a given race are considered together. `max("lap").over(w_last_lap)` computes the highest lap number (i.e. the final lap) and adds a new column called `max_lap` with the value of the last lap. I then filtered with `filter(col("lap") == col("max_lap"))` to only keep rows where the `lap` value equals the last lap. This way the position represents their finishing position.


In [0]:
# add driverId back and compute avg pit stop
df_avg_pit = df_clean.groupBy("driverId", "driverRef", "raceId").agg(
    avg("duration_seconds").alias("avg_pitstop_time")
)

# cast finishing_position to int, then join and sort — no rank() needed
df_ranked = df_avg_pit.join(df_last_lap, on=["raceId", "driverId"]).withColumn(
    "finishing_position", col("finishing_position").cast("int")).orderBy(
        "raceId", "finishing_position")

df_ranked.select("raceId", "driverRef", "finishing_position", "avg_pitstop_time").show(20)


+------+---------------+------------------+----------------+
|raceId|      driverRef|finishing_position|avg_pitstop_time|
+------+---------------+------------------+----------------+
|  1000|       hamilton|                 1|           21.48|
|  1000|         bottas|                 2|           21.34|
|  1000|         vettel|                 3|           23.11|
|  1000|      raikkonen|                 4|           23.15|
|  1000|          gasly|                 5|           21.68|
|  1000|kevin_magnussen|                 6|           25.13|
|  1000|          sainz|                 7|           21.91|
|  1000|brendon_hartley|                 8|           21.83|
|  1000|     hulkenberg|                 9|           22.31|
|  1000|       grosjean|                10|           21.73|
|  1000|      ricciardo|                11|           21.36|
|  1000|         alonso|                12|            21.8|
|  1000|      vandoorne|                13|           21.73|
|  1000|           ocon|


In the code above I re-used `df_clean` from question, then joined with last lap data so that each driver's average pit stop duration each race is paired with their finishing position in that race. After joining, finishing position is cast to integer and the dataframe is sorted by `raceId` then `finishing_position`. I did not call `rank()` because `finishing_position` is already an ordinal rank, and sorting by it directly produces a rank-ordered result within each race.


## 3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.


The code column in df_drivers contains 3-letter driver codes (e.g., ALO for Alonso). Some drivers have this field missing/null. The logic is to generate the missing codes from the driver's surname. Judging from existing data, the naming convention likely is to take the first 3 letters of the surname or the initials (e.g. de la Rosa as DLR), or in some cases, first letter of first name + first two letters of last name (e.g. Franck Montagny -> FMO). To begin I will check which rows have null codes, fill them in using the surname, and update the dataframe. If that generated code already exists in the dataset (either as a real or previously generated code), pick first letter of forename + first 2 letters of surname. For multi-part surnames take initials (e.g. "de la Rosa" -> "DLR").

In [0]:
# inspect how many drivers have missing codes
df_drivers.filter(col("code").isNull() | (col("code") == "\\N")).show(5)


+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+
|driverId|  driverRef|number|code| forename|    surname|       dob|nationality|                 url|
+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+
|      43|      matta|    \N|  \N|Cristiano|   da Matta|1973-09-19|  Brazilian|http://en.wikiped...|
|      44|      panis|    \N|  \N|  Olivier|      Panis|1966-09-02|     French|http://en.wikiped...|
|      45|    pantano|    \N|  \N|  Giorgio|    Pantano|1979-02-04|    Italian|http://en.wikiped...|
|      46|      bruni|    \N|  \N|Gianmaria|      Bruni|1981-05-30|    Italian|http://en.wikiped...|
|      47|baumgartner|    \N|  \N|    Zsolt|Baumgartner|1981-01-01|  Hungarian|http://en.wikiped...|
+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+
only showing top 5 rows


In [0]:
# separate existing codes vs. missing codes
df_valid = df_drivers.filter(col("code") != "\\N")
df_missing = df_drivers.filter(col("code") == "\\N")

# collect existing codes to check against
existing_codes = set(
    row["code"] for row in df_valid.select("code").collect()
)

def generate_code(forename, surname):
    # clean special characters (apostrophes, hyphens, etc.) before processing
    surname_clean = re.sub(r"[^a-zA-Z\s]", "", surname).strip()
    forename_clean = re.sub(r"[^a-zA-Z\s]", "", forename).strip()

    # split cleaned surname into parts
    parts = surname_clean.split()

    # compound surname (3+ words) - initials of first three words
    if len(parts) > 2:
        code = "".join(p[0] for p in parts[:3]).upper()

    # two-word surname - first letter of forename + initials of both parts
    elif len(parts) == 2:
        code = (forename_clean[0] + "".join(p[0] for p in parts)).upper()

    # simple surname - first 3 letters
    elif len(surname_clean) >= 3:
        code = surname_clean[:3].upper()

        # alternative if there's a match: first letter of forename + first 2 of surname
        if code in existing_codes:
            code = (forename_clean[0] + surname_clean[:2]).upper()
            
    else:
        # very short surname fallback
        code = (forename_clean[0] + surname_clean[:2]).upper()

    return code

generate_code_udf = udf(generate_code, StringType())


First I inspected columns with missing driver codes by calling `filter` on the `code` column, then separated existing codes vs. missing codes from the original drivers dataframe by making two individual copy dataframes, and collected existing codes to check against cases of duplicates with `collect`. 
I created a function `generate_code` to check if code is valid with the following logic:`strip` and `split` splits a compound surname. If the surname contains more than two parts, the first three initial characters are selected and joined, then converted to upper case with `upper` (like DLR for de la Rosa). Surnames with two parts, have the first letter of their first name attached to the two parts' initials. Surnames that don't have parts, if having more than three characters (this disregards last names that are extremely short, but since they are usually rarer cases, I will manually handle them) get their first 3 letters, and in case of duplicates, use first letter of forename  + first 2 of surname (like FMO for Franck Montagny). The exceptions to this rule should be rare so instead of creating more rules I decide I will code them manually. 


In [0]:
# apply to missing codes
df_missing = df_missing.withColumn("code", generate_code_udf(col("forename"), col("surname")))

# check if code is valid
df_missing = df_missing.withColumn("invalid", when(col("code").isin(existing_codes), True).otherwise(False))

df_missing.show(5)

+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+-------+
|driverId|  driverRef|number|code| forename|    surname|       dob|nationality|                 url|invalid|
+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+-------+
|      43|      matta|    \N| CDM|Cristiano|   da Matta|1973-09-19|  Brazilian|http://en.wikiped...|  false|
|      44|      panis|    \N| PAN|  Olivier|      Panis|1966-09-02|     French|http://en.wikiped...|  false|
|      45|    pantano|    \N| PAN|  Giorgio|    Pantano|1979-02-04|    Italian|http://en.wikiped...|  false|
|      46|      bruni|    \N| BRU|Gianmaria|      Bruni|1981-05-30|    Italian|http://en.wikiped...|  false|
|      47|baumgartner|    \N| BAU|    Zsolt|Baumgartner|1981-01-01|  Hungarian|http://en.wikiped...|  false|
+--------+-----------+------+----+---------+-----------+----------+-----------+--------------------+-------+
only showing top 5 

I applied the function to the `df_missing` dataframe, then checked the populated `df_missing` dataframe against the `existing_codes` to see whether some codes I generated already exist. I coded the existence status with a column containing a boolean value.

In [0]:
display(df_missing.filter(col("invalid") == True))

driverId,driverRef,number,code,forename,surname,dob,nationality,url,invalid
105,alboreto,\N,MAL,Michele,Alboreto,1956-12-23,Italian,http://en.wikipedia.org/wiki/Michele_Alboreto,true
452,monarch,\N,TMO,Thomas,Monarch,1945-09-03,American,http://en.wikipedia.org/wiki/Talk:1963_Mexican_Grand_Prix#Who_was_Thomas_Monarch.3F,true
787,harrison,\N,CHA,Cuth,Harrison,1906-07-06,British,http://en.wikipedia.org/wiki/Cuth_Harrison,true


Here it can be seen that my rule only yielded three duplicates, which is manageable.

In [0]:
# manually fix the 3 collision cases by driverId
df_missing_fixed = df_missing.withColumn(
    "code",
    when(col("driverId") == 105, "MAB")  # Michele Alboreto
    .when(col("driverId") == 452, "TMO")  # Thomas Monarch - already unique, double check
    .when(col("driverId") == 787, "CUH")  # Cuth Harrison
    .otherwise(col("code"))
)

# now all rows are valid, drop the invalid column and union 
df_drivers_fixed = df_valid.unionByName(df_missing_fixed.drop("invalid"))


df_drivers_fixed.show(5)

+--------+----------+------+----+--------+----------+----------+-----------+--------------------+
|driverId| driverRef|number|code|forename|   surname|       dob|nationality|                 url|
+--------+----------+------+----+--------+----------+----------+-----------+--------------------+
|       1|  hamilton|    44| HAM|   Lewis|  Hamilton|1985-01-07|    British|http://en.wikiped...|
|       2|  heidfeld|    \N| HEI|    Nick|  Heidfeld|1977-05-10|     German|http://en.wikiped...|
|       3|   rosberg|     6| ROS|    Nico|   Rosberg|1985-06-27|     German|http://en.wikiped...|
|       4|    alonso|    14| ALO|Fernando|    Alonso|1981-07-29|    Spanish|http://en.wikiped...|
|       5|kovalainen|    \N| KOV|  Heikki|Kovalainen|1981-10-19|    Finnish|http://en.wikiped...|
+--------+----------+------+----+--------+----------+----------+-----------+--------------------+
only showing top 5 rows


I merged (the completed rows from) `df_missing` with the existing codes dataframe as those are guaranteed to have no duplicates. Then I hand-coded the driver codes for the problematic cases, then for the next step will check again whether my manually-code value has a duplicate. 

In [0]:
display(df_drivers[(col('code')=='MAB') | (col('code')=='TMO') | (col('code')=='CUH')])
# since there already exists a TMO, change the code for Thomas Monarch to TMN


driverId,driverRef,number,code,forename,surname,dob,nationality,url
33,monteiro,\N,TMO,Tiago,Monteiro,1976-07-24,Portuguese,http://en.wikipedia.org/wiki/Tiago_Monteiro


In [0]:
display(df_drivers[(col('code')=='TMN')])
# no existing code - safe to proceed 

df_drivers_fixed = df_drivers_fixed.withColumn(
    "code",
    when(col("driverId") == 452, "TMN").otherwise(col("code")
))


driverId,driverRef,number,code,forename,surname,dob,nationality,url


In [0]:
df_drivers_fixed.filter(col("code").isNull() | (col("code") == "\\N")).show()

+--------+---------+------+----+--------+-------+---+-----------+---+
|driverId|driverRef|number|code|forename|surname|dob|nationality|url|
+--------+---------+------+----+--------+-------+---+-----------+---+
+--------+---------+------+----+--------+-------+---+-----------+---+



I hand-coded again and checked against the fixed df_drivers dataframe and proceeded after making sure there is no duplicate for the new driver name I created. The fixed drivers dataframe now should have no missing driver codes and no duplicates. 

## 4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".


For this question there may be two ways around: one is the simplest, the other more precise. 
The first way calculates age as year of race - birth year. Though this definition may be flawed as they may calculate someone as one year older when their birthday hasn't occurred yet that year.
The second definition calculates age as the number of complete years that have passed since the date of birth, accounting for whether they've had a birthday that year.

In [0]:
# get dates for each race

df_races = df_races.select('raceId', 'date').withColumn("race_date", col("date").cast("date"))

df_races.show(5)

+------+----------+----------+
|raceId|      date| race_date|
+------+----------+----------+
|     1|2009-03-29|2009-03-29|
|     2|2009-04-05|2009-04-05|
|     3|2009-04-19|2009-04-19|
|     4|2009-04-26|2009-04-26|
|     5|2009-05-10|2009-05-10|
+------+----------+----------+
only showing top 5 rows


I selected the `raceId` and `date` columns from the `races` dataframe, casted the type for the date as `'date'`.

In [0]:
# get one row per driver per race from laptimes, then join with drivers and race dates
df_driver_race = df_laptimes.select("raceId", "driverId") \
    .dropDuplicates(["raceId", "driverId"]) \
    .join(df_drivers.select("driverId", "driverRef", "forename", "surname", "nationality", "dob"), on="driverId") \
    .join(df_races, on="raceId")

df_driver_race.show(5)


+------+--------+---------+--------+---------+-----------+----------+----------+----------+
|raceId|driverId|driverRef|forename|  surname|nationality|       dob|      date| race_date|
+------+--------+---------+--------+---------+-----------+----------+----------+----------+
|   842|       3|  rosberg|    Nico|  Rosberg|     German|1985-06-27|2011-04-10|2011-04-10|
|   846|     155|kobayashi|   Kamui|Kobayashi|   Japanese|1986-09-13|2011-05-29|2011-05-29|
|   847|      17|   webber|    Mark|   Webber| Australian|1976-08-27|2011-06-12|2011-06-12|
|   848|      16|    sutil|  Adrian|    Sutil|     German|1983-01-11|2011-06-26|2011-06-26|
|   849|      17|   webber|    Mark|   Webber| Australian|1976-08-27|2011-07-10|2011-07-10|
+------+--------+---------+--------+---------+-----------+----------+----------+----------+
only showing top 5 rows


Here I deduplicated `df_laptimes` down to one row per driver-race combination before joining. I then joined with `df_drivers` to bring in driver info including date of birth, and with `df_races` to bring in the race date.

In [0]:
# compute age at the time of the race (complete years elapsed since dob)
df_driver_race = df_driver_race.withColumn(
    "age",
    floor(datediff(col("race_date"), col("dob").cast("date")) / 365)
)


I define age here as the number of complete years between a driver's dob and the race date. I use `floor(datediff/365)` to count only fully elapsed years. 

In [0]:
# window to find min/max age per race
w = Window.partitionBy("raceId")

df_age_ranked = df_driver_race \
    .withColumn("youngest_age", min("age").over(w)) \
    .withColumn("oldest_age", max("age").over(w))

# filter for youngest and oldest, label, then union
df_youngest = df_age_ranked.filter(col("age") == col("youngest_age")) \
    .select("raceId", "driverRef", "age") \
    .withColumn("category", lit("youngest"))

df_oldest = df_age_ranked.filter(col("age") == col("oldest_age")) \
    .select("raceId", "driverRef", "age") \
    .withColumn("category", lit("oldest"))

df_result = df_youngest.unionByName(df_oldest).orderBy("raceId", "category")
df_result.show(5)


+------+-----------+---+--------+
|raceId|  driverRef|age|category|
+------+-----------+---+--------+
|     1|barrichello| 36|  oldest|
|     1| fisichella| 36|  oldest|
|     1|      buemi| 20|youngest|
|    10|barrichello| 37|  oldest|
|    10|alguersuari| 19|youngest|
+------+-----------+---+--------+
only showing top 5 rows


A `Window` partitioned by `raceId` computes the minimum and maximum age across all drivers in each race. I then filter separately for drivers matching the youngest and oldest age in their race, label each with a `category` column, and unioned the two dataframes together. The result is sorted by `raceId` then category, showing the oldest and youngest driver side by side for each race. In cases where multiple drivers share the same minimum or maximum age in a race, all tied drivers are included in the output. 

## 5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

For each race, it is asked how many times each driver has finished 1st, 2nd, or 3rd up to and including that race. This is a cumulative count, so I need a Window function ordered by `raceId` for each driver. I use sum of a boolean condition (1 if position matches, 0 otherwise) over a window that grows as races progress, which outputs a running total of wins, 2nd places, and 3rd places at any given point in the season.


In [0]:
# reuse df_last_lap from Q2, casting finishing-position column into int
df_finishing = df_last_lap.withColumn(
    "finishing_position", col("finishing_position").cast("int")
)

# code binary values for three situations
df_podium_flags = df_finishing.withColumn(
    "is_first", when(col("finishing_position") == 1, 1).otherwise(0)
).withColumn(
    "is_second", when(col("finishing_position") == 2, 1).otherwise(0)
).withColumn(
    "is_third", when(col("finishing_position") == 3, 1).otherwise(0)
)

df_podium_flags.show(5)

+------+--------+------------------+--------+---------+--------+
|raceId|driverId|finishing_position|is_first|is_second|is_third|
+------+--------+------------------+--------+---------+--------+
|     1|       1|                 9|       0|        0|       0|
|     1|      13|                 3|       0|        0|       1|
|     1|      18|                 1|       1|        0|       0|
|     1|       7|                16|       0|        0|       0|
|    10|       6|                 9|       0|        0|       0|
+------+--------+------------------+--------+---------+--------+
only showing top 5 rows


I reused the `df_last_lap` dataset from Q2, casting finishing-position as integer, to get the finishing position for each driver each race. 
I then use `when` statement to create binary columns where the value is '1' when the finishing position corresponds to the column name. 

In [0]:
# create cumulative window partitioned by driver and ordered by race id
w_cumulative = Window.partitionBy("driverId") \
    .orderBy("raceId") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# add columns logging cumulative wins, 2nd and 3rd
df_podiums = df_podium_flags \
    .withColumn("total_wins", sum("is_first").over(w_cumulative)) \
    .withColumn("total_2nd", sum("is_second").over(w_cumulative)) \
    .withColumn("total_3rd", sum("is_third").over(w_cumulative))

# join with driver names
df_result = df_podiums.join(
    df_drivers.select("driverId", "driverRef"), on="driverId"
).select("raceId", "driverRef", "finishing_position", "total_wins", "total_2nd", "total_3rd") \
 .orderBy("raceId", "driverRef")


df_result.show(5)

+------+-----------+------------------+----------+---------+---------+
|raceId|  driverRef|finishing_position|total_wins|total_2nd|total_3rd|
+------+-----------+------------------+----------+---------+---------+
|     1|     alonso|                14|         0|        0|        0|
|     1|barrichello|                 7|         0|        0|        0|
|     1|   bourdais|                16|         0|        0|        0|
|     1|      buemi|                11|         0|        0|        0|
|     1|     button|                 1|         1|        0|        0|
+------+-----------+------------------+----------+---------+---------+
only showing top 5 rows


The cumulative Window is partitioned by `driverId` and ordered by `raceId`, with `rowsBetween(unboundedPreceding, currentRow)` meaning all rows from the top of the df (i.e. beginning with the earliest races) up to the current race. I then `sum` over this window, which returns a running total that grows with each race, recorded under new columns `'total_wins'`, `'total_2nd'`, and `'total_3rd'`. I joined `driverRef` back in the dataframe for readability, then sorted by race and driver.

## 6.Continue exploring the data by answering your own question.


I want to explore which nations have the most capable drivers, defined as the number of total wins per driver per nationality. For this, I will need the `nationality` column from df_drivers, and the accumulated `df_podium_flags` from the previous quetsion. I will join the `nationality` column with `df_podium_flags`, then extract the column for first places, and aggregate by the sum of first places. To account for historically active nations/nationalities, I will normalize the total wins by the number of drivers from each nationality. 


In [0]:
# merge drivers ref and nationality with podium_flags dataframe from last question
df_result_nat = df_drivers.select("driverId", "driverRef", "nationality").join(
    df_podium_flags.select("driverId", "is_first"), on = ["driverId"])

df_result_nat.show(5)

+--------+---------+-----------+--------+
|driverId|driverRef|nationality|is_first|
+--------+---------+-----------+--------+
|       1| hamilton|    British|       0|
|      13|    massa|  Brazilian|       0|
|      18|   button|    British|       1|
|       7| bourdais|     French|       0|
|       6| nakajima|   Japanese|       0|
+--------+---------+-----------+--------+
only showing top 5 rows


I merged `df_drivers` with `df_podium_flags` from the previous quetsion with `join` on `driverId`, and selected my main columns of interest: `nationality` and `is_first`. 

In [0]:
# group by nationality, calculate total wins, 2nd, and 3rd place

df_nat_agg = df_result_nat.groupBy("nationality").agg(
    sum("is_first").alias("total_wins")).orderBy("total_wins", ascending=False)

df_nat_agg.show(5)

+-----------+----------+
|nationality|total_wins|
+-----------+----------+
|     German|       154|
|    British|       132|
|    Finnish|        66|
|      Dutch|        47|
|    Spanish|        35|
+-----------+----------+
only showing top 5 rows


I called `groupBy` on the merged dataframe, selected `nationality`, and used `agg` to calculate the sum of first places for each nation, creating a new `total_wins` column to record the sum. 
However, as certain nations may be historically more active in f1 races, I will normalize the results by the unique number of drivers from each nationality. 


In [0]:
# count drivers per nationality
df_driver_count = df_result_nat.groupBy("nationality").agg(countDistinct("driverId").alias("num_drivers"))

df_driver_count.show(5)

+-----------+-----------+
|nationality|num_drivers|
+-----------+-----------+
|   Japanese|          9|
|    Italian|         12|
|    Belgian|          2|
|    Spanish|          6|
|  Hungarian|          1|
+-----------+-----------+
only showing top 5 rows


I called `groupBy` and grouped by `nationality`, similar to the previous code block, on `df_result_nat` which contained `driverId`, then counted the distinct IDs with the `countDistinct` method, renaming the aggregared value as `'num_drivers'`.

In [0]:
# combine with nat_agg then calculate normalized win counts by dividing total win counts by number of drivers

df_win_normalized = df_driver_count.join(df_nat_agg, on = ["nationality"]).withColumn("normalized_wins", col("total_wins")/col("num_drivers")).orderBy("normalized_wins", ascending = False)

df_win_normalized.show(5)

+-----------+-----------+----------+------------------+
|nationality|num_drivers|total_wins|   normalized_wins|
+-----------+-----------+----------+------------------+
| Monegasque|          1|        16|              16.0|
|    Finnish|          5|        66|              13.2|
|     German|         13|       154|11.846153846153847|
|  Colombian|          1|        11|              11.0|
|      Dutch|          6|        47| 7.833333333333333|
+-----------+-----------+----------+------------------+
only showing top 5 rows


I joined the `df_nat_agg` dataframe with the `df_driver_count` dataframe on `nationality`, added a column called `normalized_wins`, calculated by dividing total wins by the number of drivers. After sorting the `normalized_wins` it can be seen that the Mongasque drivers are the strongest overall, with highest number of wins per driver. (However, it's also notable that they only have one driver), followed by Finnish and German.
An alternative metric would be `total_wins / total_race_starts` (win rate per entry), but here I prioritized talent pool strength over win rate.
